In [2]:
!pip install matplotlib

In [1]:
"""
Polygon.io Options Historical Data & Chart Generator
Unity Software (U) $40 Call - Expiring Jan 15, 2027

Requirements: pip install requests matplotlib pandas
"""

import requests
import json
from datetime import datetime
import sys

# Your API Key
API_KEY = "mxHpmdO4wzkVJhzExKhIfbXbUDr0OmCW"

# Unity $40 Call expiring 1/15/27
# Format: O:{underlying}{YYMMDD}{C/P}{strike*1000 padded to 8 digits}
OPTIONS_TICKER = "O:U270115C00040000"

def get_options_contracts(underlying="U", expiration_gte="2027-01-01", expiration_lte="2027-01-31"):
    """First, let's verify the contract exists and get its details"""
    url = "https://api.polygon.io/v3/reference/options/contracts"
    params = {
        "underlying_ticker": underlying,
        "expiration_date.gte": expiration_gte,
        "expiration_date.lte": expiration_lte,
        "strike_price": 40,
        "contract_type": "call",
        "limit": 10,
        "apiKey": API_KEY
    }
    
    response = requests.get(url, params=params)
    return response.json()


def get_historical_aggregates(ticker, from_date, to_date, timespan="day", multiplier=1):
    """
    Get historical OHLC data for an options contract
    
    Parameters:
    - ticker: Options ticker (e.g., O:U270115C00040000)
    - from_date: Start date (YYYY-MM-DD)
    - to_date: End date (YYYY-MM-DD)
    - timespan: minute, hour, day, week, month, quarter, year
    - multiplier: Size of timespan (e.g., 5 with minute = 5-minute bars)
    """
    url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/{multiplier}/{timespan}/{from_date}/{to_date}"
    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": 50000,
        "apiKey": API_KEY
    }
    
    response = requests.get(url, params=params)
    return response.json()


def get_daily_open_close(ticker, date):
    """Get single day OHLC for an options contract"""
    url = f"https://api.polygon.io/v1/open-close/{ticker}/{date}"
    params = {"apiKey": API_KEY}
    
    response = requests.get(url, params=params)
    return response.json()


def create_chart(data):
    """Create a price chart from the historical data"""
    try:
        import matplotlib.pyplot as plt
        import matplotlib.dates as mdates
        import pandas as pd
        
        if not data.get("results"):
            print("No data available to chart")
            return
        
        # Convert to DataFrame
        df = pd.DataFrame(data["results"])
        df["date"] = pd.to_datetime(df["t"], unit="ms")
        df.set_index("date", inplace=True)
        
        # Create the chart
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})
        
        # Price chart (candlestick-style using line)
        ax1.fill_between(df.index, df["l"], df["h"], alpha=0.3, color="blue", label="High-Low Range")
        ax1.plot(df.index, df["c"], color="blue", linewidth=2, label="Close Price")
        ax1.plot(df.index, df["o"], color="green", linewidth=1, linestyle="--", alpha=0.7, label="Open Price")
        
        ax1.set_title(f"Unity (U) $40 Call - Exp 1/15/2027\nHistorical Price Chart", fontsize=14, fontweight="bold")
        ax1.set_ylabel("Option Price ($)")
        ax1.legend(loc="upper left")
        ax1.grid(True, alpha=0.3)
        ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax1.xaxis.set_major_locator(mdates.AutoDateLocator())
        
        # Volume chart
        ax2.bar(df.index, df["v"], color="purple", alpha=0.7)
        ax2.set_ylabel("Volume")
        ax2.set_xlabel("Date")
        ax2.grid(True, alpha=0.3)
        ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig("unity_options_chart.png", dpi=150)
        plt.show()
        print("\nChart saved as 'unity_options_chart.png'")
        
    except ImportError:
        print("\nTo create charts, install: pip install matplotlib pandas")
        print("Data is still available in JSON format above.")


def main():
    print("=" * 70)
    print("Polygon.io Options Historical Data Fetcher")
    print("=" * 70)
    
    # Step 1: Check if contract exists
    print("\n📋 Step 1: Checking for Unity options contracts expiring Jan 2027...")
    contracts = get_options_contracts()
    
    if contracts.get("results"):
        print(f"✅ Found {len(contracts['results'])} matching contract(s):")
        for c in contracts["results"]:
            print(f"   - {c.get('ticker')}: ${c.get('strike_price')} {c.get('contract_type').upper()}, expires {c.get('expiration_date')}")
        
        # Use the actual ticker from the response if different
        actual_ticker = contracts["results"][0].get("ticker", OPTIONS_TICKER)
    else:
        print(f"⚠️  No contracts found for Jan 2027. This might mean:")
        print("   - The contract hasn't been listed yet (LEAPS often list ~2 years out)")
        print("   - Using the constructed ticker: " + OPTIONS_TICKER)
        actual_ticker = OPTIONS_TICKER
    
    # Step 2: Get historical data
    print(f"\n📊 Step 2: Fetching historical data for {actual_ticker}...")
    
    # Get data from when contracts might have started trading to now
    historical_data = get_historical_aggregates(
        ticker=actual_ticker,
        from_date="2024-01-01",
        to_date=datetime.now().strftime("%Y-%m-%d"),
        timespan="day",
        multiplier=1
    )
    
    if historical_data.get("status") == "OK" and historical_data.get("results"):
        print(f"✅ Found {historical_data.get('resultsCount', 0)} data points!")
        print("\n📈 Sample Data (first 10 bars):")
        print("-" * 70)
        print(f"{'Date':<12} {'Open':>10} {'High':>10} {'Low':>10} {'Close':>10} {'Volume':>10}")
        print("-" * 70)
        
        for bar in historical_data["results"][:10]:
            date = datetime.fromtimestamp(bar["t"] / 1000).strftime("%Y-%m-%d")
            print(f"{date:<12} ${bar['o']:>9.2f} ${bar['h']:>9.2f} ${bar['l']:>9.2f} ${bar['c']:>9.2f} {bar['v']:>10}")
        
        print("-" * 70)
        print(f"... and {len(historical_data['results']) - 10} more data points")
        
        # Step 3: Create chart
        print("\n📊 Step 3: Creating price chart...")
        create_chart(historical_data)
        
    else:
        print(f"⚠️  Status: {historical_data.get('status')}")
        print(f"   Message: {historical_data.get('message', 'No data available')}")
        print("\n💡 Possible reasons:")
        print("   - Contract may not have enough trading history yet")
        print("   - LEAPS contracts sometimes have limited liquidity")
        print("   - Try a different date range or a more liquid contract")
    
    # Bonus: Show API info
    print("\n" + "=" * 70)
    print("📚 Useful API Endpoints for Options:")
    print("=" * 70)
    print("""
1. Get Historical Bars (OHLC):
   GET /v2/aggs/ticker/{optionsTicker}/range/{multiplier}/{timespan}/{from}/{to}
   
2. Get Daily Open/Close:
   GET /v1/open-close/{optionsTicker}/{date}
   
3. List Options Contracts:
   GET /v3/reference/options/contracts?underlying_ticker=U
   
4. Get Real-time Quotes:
   GET /v3/quotes/{optionsTicker}
   
5. Get Trades:
   GET /v3/trades/{optionsTicker}
   
6. Get Greeks & IV (Snapshot):
   GET /v3/snapshot/options/{underlyingAsset}/{optionContract}
""")


if __name__ == "__main__":
    main()

Polygon.io Options Historical Data Fetcher

📋 Step 1: Checking for Unity options contracts expiring Jan 2027...
✅ Found 1 matching contract(s):
   - O:U270115C00040000: $40 CALL, expires 2027-01-15

📊 Step 2: Fetching historical data for O:U270115C00040000...
✅ Found 291 data points!

📈 Sample Data (first 10 bars):
----------------------------------------------------------------------
Date               Open       High        Low      Close     Volume
----------------------------------------------------------------------
2024-09-16   $     4.00 $     4.00 $     3.45 $     3.85          9
2024-09-18   $     3.95 $     3.95 $     3.80 $     3.80         11
2024-09-19   $     4.04 $     4.04 $     3.65 $     3.65          5
2024-09-20   $     3.53 $     3.95 $     3.53 $     3.95          8
2024-09-23   $     4.25 $     4.25 $     4.10 $     4.10         27
2024-09-24   $     4.50 $     4.75 $     4.25 $     4.75        157
2024-09-25   $     4.99 $     4.99 $     4.67 $     4.67         